# Use shared SFT/DAPO adapters

2026-11-24. Select a GPU runtime. This notebook performs inference only. Copy the delivered ZIP to your Drive first. It contains adapters and reports; the pinned base is downloaded on first explicit load. Respect each base license. GPU use consumes your own Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import zipfile
BUNDLE=Path("/content/drive/MyDrive/mtrl-models-2026-11-24.zip") # change to your ZIP
assert BUNDLE.is_file(), BUNDLE
with zipfile.ZipFile(BUNDLE) as z:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts and ":" not in n and "\\" not in n for n in z.namelist())
    z.extractall("/content")
ROOT=Path("/content/mtrl")
assert (ROOT/"models/registry.json").exists()


In [ ]:
%pip install -q -r /content/mtrl/setup/requirements-checkpoint-use.txt


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/"scripts/models.py"),"list"],check=True)


Choose either SFT baseline or a DAPO candidate. `recorded` uses FP32 and needs an adequately large GPU for3B. Reduced precision is practical inference, not an exact-score reproduction. This cell loads one model and does not train it.


In [ ]:
import torch
sys.path.insert(0,str(ROOT/"src"))
from mtrl.checkpoints import load_translation_model,translate
MODEL_ID="qwen25-3b-sft-r0" # or qwen35-08b-sft-r0 / qwen25-3b-dapo-shortmid
PRECISION="bfloat16" if torch.cuda.get_device_capability()[0]>=8 else "float16"
model,tokenizer,info=load_translation_model(MODEL_ID,root=ROOT,device="cuda",precision=PRECISION)


In [ ]:
translate(model,tokenizer,"Put the book on the table.",max_new_tokens=320)


For8candidates use `translate(...,samples=8,temperature=.7)`. For new training, start a fresh load with `trainable=True` in your own training script; loading that flag alone does not perform updates. Do not treat this as exact optimizer/RNG resume. Do not load a second3B model on a small GPU before releasing the first.
